# Module 2 Homework — Stock Markets Analytics Zoomcamp 2026

Lesson reference: [`02-dataframe-analysis/[2026]_Module_02_Colab_Working_with_the_data.ipynb`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/02-dataframe-analysis/%5B2026%5D_Module_02_Colab_Working_with_the_data.ipynb)

Homework spec: [`cohorts/2026/homework2.md`](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework2.md)
Submission form: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw02

## Question 1: [IPO] Withdrawn IPOs by Company Type

**Main task:** What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

Source: iposcoop.com "Recently Filed" table, scraped with `pandas.read_html()`. The table is a rolling list, so we keep withdrawn rows with a File Date before 2026-09-11 (31 rows vs. the spec's 32 — within the variation the spec says to expect).

In [1]:
"""
Question 1: [IPO] Withdrawn IPOs by Company Type

What is the total withdrawn IPO value (in $ millions) for the company class
with the highest total withdrawal value?

Note: the iposcoop "recently filed" table is a rolling list, so it keeps
growing after the homework was written. We keep only withdrawn rows with a
File Date before 2026-09-11 to match the question's "before Sep 11, 2026".
"""
import io

import numpy as np
import pandas as pd
import requests

URL = "https://www.iposcoop.com/ipos-recently-filed/"
CUTOFF = pd.Timestamp("2026-09-11")

# order matters -- first match wins
COMPANY_TYPE_RULES = [
    (["Technologies"], "Technologies"),
    (["Acquisition Corp", "Acquisition Corporation", "Corp"], "Acquisition Corp"),
    (["Inc", "Incorporated"], "Inc."),
    (["Group"], "Group"),
    (["Ltd", "Limited"], "Limited"),
    (["Holdings", "Holding"], "Holdings"),
]


def load_filed_ipos():
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(URL, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))[0]


def classify_company(name):
    for patterns, label in COMPANY_TYPE_RULES:
        if any(p in name for p in patterns):
            return label
    return "Other"


def parse_price(value):
    # '$8.00' -> 8.0, '-' / NaN -> NaN
    if pd.isna(value) or str(value).strip() == "-":
        return np.nan
    return float(str(value).replace("$", "").replace(",", ""))


def to_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
        errors="coerce",
    )


def main():
    df = load_filed_ipos()
    df["File Date"] = pd.to_datetime(df["File Date"])

    withdrawn = df[(df["Expected To Trade"] == "Withdrawn") & (df["File Date"] < CUTOFF)].copy()
    print(f"Withdrawn IPOs filed before {CUTOFF.date()}: {len(withdrawn)} (spec expects ~32)")

    withdrawn["Company Type"] = withdrawn["Company"].apply(classify_company)
    withdrawn["Avg_price"] = (
        withdrawn["Price Low"].apply(parse_price) + withdrawn["Price High"].apply(parse_price)
    ) / 2
    withdrawn["Shares (millions)"] = to_numeric(withdrawn["Shares (millions)"])
    withdrawn["Est $ Vol (millions)"] = to_numeric(withdrawn["Est $ Vol (millions)"])

    withdrawn["Shares_offered_value"] = (
        withdrawn["Shares (millions)"] * withdrawn["Avg_price"]
    ).fillna(withdrawn["Est $ Vol (millions)"])

    print(
        withdrawn[["Company", "Company Type", "Shares (millions)", "Avg_price",
                   "Est $ Vol (millions)", "Shares_offered_value"]].to_string(index=False)
    )

    by_type = withdrawn.groupby("Company Type")["Shares_offered_value"].agg(["count", "sum"])
    by_type = by_type.sort_values("sum", ascending=False)
    print("\nWithdrawn value by Company Type ($M):")
    print(by_type)

    top = by_type.index[0]
    print(f"\nANSWER Q1: {top} with ${by_type.loc[top, 'sum']:,.2f}M withdrawn")


main()


Withdrawn IPOs filed before 2026-09-11: 31 (spec expects ~32)
                                        Company     Company Type  Shares (millions)  Avg_price  Est $ Vol (millions)  Shares_offered_value
                Motive Technologies (Withdrawn)     Technologies               0.00        NaN                100.00              100.0000
                  Idea Tech Holding (Withdrawn)         Holdings               2.00       4.50                  9.00                9.0000
         Hornbeck Offshore Services (Withdrawn)            Other               0.00        NaN                100.00              100.0000
          Coolbit Technologies Ltd. (Withdrawn)     Technologies               5.00       4.50                 22.50               22.5000
      Timber Road Acquisition Corp. (Withdrawn) Acquisition Corp              20.00      10.00                200.00              200.0000
                   pan-Africa Corp. (Withdrawn) Acquisition Corp              12.50      10.00          

## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

**Main task:** What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?

Source: iposcoop.com 2025 pricings + daily OHLCV from `yfinance`. Volatility uses the spec's formula (`Close.rolling(30).std() * sqrt(252)`), i.e. std of the price level rather than of returns. A few flat-priced tickers get 0 volatility → infinite Sharpe, which breaks the mean but not the median.

In [2]:
"""
Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

What is the median Sharpe ratio (as of 11 September 2026) for companies that
went public before 1 September 2025?

The downloaded OHLCV data is cached to data/stocks_df.csv so Q3 can reuse the
same stocks_df without hitting yfinance again. Delete the file to re-download.
"""
import io
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

URL = "https://www.iposcoop.com/2025-pricings/"
OFFER_CUTOFF = pd.Timestamp("2025-09-01")
END_DATE = "2026-09-12"  # yfinance end is exclusive -> includes 2026-09-11
EVAL_DATE = "2026-09-11"
RISK_FREE = 0.05

CACHE = Path("data") / "stocks_df.csv"


def load_ipos_2025():
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(URL, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))[0]


def filter_ipos(ipos):
    ipos = ipos.copy()
    ipos["Offer Date"] = pd.to_datetime(ipos["Offer Date"], format="%m/%d/%Y")
    ipos["Return"] = pd.to_numeric(ipos["Return"].str.replace("%", "", regex=False), errors="coerce")
    return ipos[(ipos["Offer Date"] < OFFER_CUTOFF) & (ipos["Return"] != 0)]


def download_stocks(tickers):
    frames = []
    missing = []
    for ticker in tickers:
        try:
            df = yf.download(ticker, start="2025-01-01", end=END_DATE, progress=False, auto_adjust=False)
        except Exception:
            df = pd.DataFrame()
        if df.empty:
            missing.append(ticker)  # probably delisted / not on Yahoo
            continue
        df.columns = df.columns.get_level_values(0)
        df["Ticker"] = ticker
        frames.append(df)
    print(f"Downloaded {len(frames)} tickers, missing {len(missing)}: {missing}")
    return pd.concat(frames)


def add_features(stocks_df):
    stocks_df = stocks_df.sort_values(["Ticker", "Date"]).copy()
    close = stocks_df.groupby("Ticker")["Close"]
    stocks_df["growth_252d"] = stocks_df["Close"] / close.shift(252)
    stocks_df["volatility"] = close.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
    stocks_df["Sharpe"] = (stocks_df["growth_252d"] - RISK_FREE) / stocks_df["volatility"]
    return stocks_df


def load_stocks_df():
    """Filtered 2025 IPO tickers' daily OHLCV + features (shared with Q3)."""
    if CACHE.exists():
        stocks_df = pd.read_csv(CACHE, parse_dates=["Date"])
    else:
        ipos = filter_ipos(load_ipos_2025())
        print(f"IPOs before {OFFER_CUTOFF.date()} with non-zero return: {len(ipos)} (spec expects 148)")
        stocks_df = download_stocks(ipos["Symbol"].tolist()).reset_index()
        CACHE.parent.mkdir(exist_ok=True)
        stocks_df.to_csv(CACHE, index=False)
    return add_features(stocks_df)


def main():
    stocks_df = load_stocks_df()
    print(f"Tickers in stocks_df: {stocks_df['Ticker'].nunique()} (spec expects ~134)")

    on_date = stocks_df[stocks_df["Date"] == EVAL_DATE]
    stats = on_date[["growth_252d", "volatility", "Sharpe"]].describe()
    print(f"\nDescriptive stats on {EVAL_DATE}:")
    print(stats)

    print("\nTop 10 by Sharpe vs. their growth rank:")
    ranked = on_date.dropna(subset=["Sharpe"]).copy()
    ranked["growth_rank"] = ranked["growth_252d"].rank(ascending=False).astype(int)
    print(
        ranked.sort_values("Sharpe", ascending=False)
        .head(10)[["Ticker", "growth_252d", "volatility", "Sharpe", "growth_rank"]]
        .to_string(index=False)
    )

    print(f"\nANSWER Q2: median Sharpe on {EVAL_DATE} = {stats.loc['50%', 'Sharpe']:.4f}")


main()


IPOs before 2025-09-01 with non-zero return: 146 (spec expects 148)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}


$MJID: possibly delisted; no timezone found



1 Failed download:


['MJID']: possibly delisted; no timezone found


$DLXY: possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-12)


... (41 outputs left out)

['MCTR']: possibly delisted; no timezone found


Downloaded 131 tickers, missing 15: ['MJID', 'DLXY', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'WGRX', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


Tickers in stocks_df: 131 (spec expects ~134)

Descriptive stats on 2026-09-11:
Price  growth_252d  volatility      Sharpe
count   129.000000  130.000000  129.000000
mean      1.059676   27.781138         inf
std       3.059882   67.083691         NaN
min       0.001005    0.000000   -0.040147
25%       0.149038    2.325895    0.011942
50%       0.596026    8.122327    0.048009
75%       1.040233   24.019924    0.134248
max      33.638270  627.698106         inf

Top 10 by Sharpe vs. their growth rank:
Ticker  growth_252d  volatility       Sharpe  growth_rank
  EFTY     2.106592    0.000000          inf            8
  MAMK     6.532663    0.000000          inf            2
  MAGH     5.451613    0.000001 4.204751e+06            4
 HCMAU     1.034585    0.182827 5.385326e+00           34
  CEPF     1.024233    0.351583 2.770994e+00           36
  TMDE     0.776166    0.414673 1.751178e+00           52
  TLIH     0.989673    0.648547 1.448889e+00           38
  CEPO     1.031160    0.795

/Users/alexyun/stock-market-analytics-zoomcamp-coursework/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


## Question 3: [IPO] 'Fixed Months Holding Strategy'

**Main task:** What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

Uses Q2's `stocks_df`, 1 month = 21 trading days, entry = each ticker's first available trading day. PPCB's first row is a $0.02 pre-uplisting price, which blows up the means (12,500x) but doesn't affect the medians.

In [3]:
"""
Question 3: [IPO] 'Fixed Months Holding Strategy'

What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in
order to maximize the median growth value?

Reuses Q2's stocks_df via load_stocks_df() from the Q2 cell above.
"""
import pandas as pd


MONTHS = range(1, 13)
DAYS_PER_MONTH = 21


def add_future_growth(stocks_df):
    stocks_df = stocks_df.copy()
    close = stocks_df.groupby("Ticker")["Close"]
    for m in MONTHS:
        stocks_df[f"future_growth_{m}_m"] = close.shift(-m * DAYS_PER_MONTH) / stocks_df["Close"]
    return stocks_df


def main():
    stocks_df = add_future_growth(load_stocks_df())

    # entry point: each ticker's first trading day
    min_dates = stocks_df.groupby("Ticker")["Date"].min().reset_index().rename(columns={"Date": "min_date"})
    entries = min_dates.merge(
        stocks_df, left_on=["Ticker", "min_date"], right_on=["Ticker", "Date"], how="inner"
    )
    print(f"Entry rows (one per ticker): {len(entries)}")

    growth_cols = [f"future_growth_{m}_m" for m in MONTHS]
    stats = entries[growth_cols].describe()
    pd.set_option("display.width", 250)
    print(stats.T)

    medians = stats.loc["50%"]
    best = medians.idxmax()
    best_month = int(best.split("_")[2])
    print(f"\nANSWER Q3: hold {best_month} month(s) -> max median growth = {medians[best]:.4f}")


main()


Entry rows (one per ticker): 131
                    count       mean          std       min       25%       50%       75%           max
future_growth_1_m   131.0  96.475125  1092.037665  0.072941  0.709910  0.930622  1.114924  12500.000279
future_growth_2_m   130.0  68.493566   767.322940  0.106197  0.582405  0.891684  1.170970   8750.000196
future_growth_3_m   130.0  51.640993   575.469433  0.090147  0.449793  0.827951  1.126212   6562.500147
future_growth_4_m   130.0  22.870282   248.776601  0.063927  0.404543  0.736039  1.193000   2837.500063
future_growth_5_m   130.0  54.658381   611.660450  0.032245  0.431361  0.694830  1.091084   6975.000156
future_growth_6_m   130.0  64.501868   723.482471  0.022075  0.315944  0.713458  1.046022   8250.000184
future_growth_7_m   130.0  59.720109   668.665978  0.013229  0.254490  0.660986  1.018706   7625.000170
future_growth_8_m   130.0  21.620850   235.633979  0.015130  0.200089  0.603984  1.033314   2687.500060
future_growth_9_m   130.0  20.1

## Question 4: [Strategy] Simple RSI-Based Trading Strategy

**Main task:** What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?

Source: the course-provided brotli-compressed parquet (via `gdown`). Trade count, average return, and win rate all match the spec exactly.

In [4]:
"""
Question 4: [Strategy] Simple RSI-Based Trading Strategy

What is the total profit (in $ thousands) you would have earned by investing
$1000 every time a stock was oversold (RSI < 30)?

The parquet file is cached to data/data.parquet after the first download.
"""
from pathlib import Path

import gdown
import pandas as pd

FILE_ID = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
DATA = Path("data") / "data.parquet"

RSI_THRESHOLD = 30
START, END = "2000-01-01", "2025-06-01"
BET = 1000


def load_data():
    if not DATA.exists():
        DATA.parent.mkdir(exist_ok=True)
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", str(DATA), quiet=False)
    return pd.read_parquet(DATA, engine="pyarrow")


def main():
    df = load_data()

    selected_df = df[
        (df["rsi"] < RSI_THRESHOLD) & (df["Date"] >= START) & (df["Date"] <= END)
    ]
    returns = selected_df["growth_future_30d"]

    print(f"Trades (RSI < {RSI_THRESHOLD}, {START} to {END}): {len(selected_df)} (spec: 5,206)")
    print(f"Average 30d return: {(returns.mean() - 1) * 100:.2f}% (spec: 1.26%)")
    print(f"Win rate: {(returns > 1).mean() * 100:.2f}% (spec: 55.13%)")

    net_income = BET * (returns - 1).sum()
    print(f"\nANSWER Q4: net income = ${net_income:,.2f} (~{net_income / 1000:.1f}K)")


main()


Trades (RSI < 30, 2000-01-01 to 2025-06-01): 5206 (spec: 5,206)
Average 30d return: 1.26% (spec: 1.26%)
Win rate: 55.13% (spec: 55.13%)

ANSWER Q4: net income = $65,805.59 (~65.8K)


## Question 5: Predicting a Positive-Return IPO (Optional)

Free-text: how would you change the IPO strategy to increase profitability?

In [5]:
"""
Question 5: [Exploratory, Optional] Predicting a Positive-Return IPO

Free-text answer: how would you change the IPO strategy to increase
profitability?
"""

ANSWER = """
From Q2/Q3, buying every 2025 IPO and holding it loses money for the typical
stock: the median growth falls from ~0.93 after 1 month to ~0.5 after a
year. So the fix is mostly about being selective and not holding blindly:

- Filter before buying -- instead of buying every IPO, only take ones with
  signals known at IPO time: offer size (tiny $4-$5 microcap deals did
  worst), whether it priced above/within its range, first-day pop vs.
  offer price, industry, and the underwriters. Train a simple classifier
  (e.g. logistic regression or a tree model) on past years' IPOs to predict
  "positive 3-month return" and only buy the top-scored ones.
- Don't buy on day 1 -- IPOs tend to drift down after the first-day hype,
  so waiting for the price to settle (or for the lock-up expiry around
  6 months, when insiders can sell) could give a better entry.
- Exit rules instead of a fixed holding period -- use a stop-loss and a
  take-profit so the few big winners run while the many losers are cut
  early.
- Short or avoid the weakest group -- if the model can spot IPOs likely to
  fall, those can be skipped (or shorted, if borrow is available).
- Clean the data first -- drop pre-IPO OTC prices (like PPCB's $0.02 row)
  and SPAC units stuck around $10 so they don't distort the stats.
"""


def main():
    print(ANSWER.strip())


main()


From Q2/Q3, buying every 2025 IPO and holding it loses money for the typical
stock: the median growth falls from ~0.93 after 1 month to ~0.5 after a
year. So the fix is mostly about being selective and not holding blindly:

- Filter before buying -- instead of buying every IPO, only take ones with
  signals known at IPO time: offer size (tiny $4-$5 microcap deals did
  worst), whether it priced above/within its range, first-day pop vs.
  offer price, industry, and the underwriters. Train a simple classifier
  (e.g. logistic regression or a tree model) on past years' IPOs to predict
  "positive 3-month return" and only buy the top-scored ones.
- Don't buy on day 1 -- IPOs tend to drift down after the first-day hype,
  so waiting for the price to settle (or for the lock-up expiry around
  6 months, when insiders can sell) could give a better entry.
- Exit rules instead of a fixed holding period -- use a stop-loss and a
  take-profit so the few big winners run while the many losers are cu

## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Company class with most withdrawn IPO value | **Acquisition Corp** (~$500M) |
| Q2 — Median Sharpe ratio on 2026-09-11 | **~0.048** |
| Q3 — Optimal holding period (max median growth) | **1 month** (median growth ~0.93) |
| Q4 — Net income from RSI < 30 strategy | **~$65.8K** ($65,805.59) |
| Q5 — Improving IPO profitability | Select IPOs with a model, wait out post-IPO drift, use stop-loss/take-profit exits — see `q5.py` |